In [ ]:
# !pip install tensorboardX

import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
if use_cuda:
    torch.cuda.manual_seed_all(42)

print("Using device:", device)

## Dataloaders
train_dataset = datasets.CIFAR10(
    'cifar10_data/', train=True, download=True,
    transform=transforms.Compose([transforms.ToTensor()])
)
test_dataset = datasets.CIFAR10(
    'cifar10_data/', train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor()])
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=2, pin_memory=use_cuda
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=2, pin_memory=use_cuda
)


Using device: cuda


100%|██████████| 170M/170M [35:06<00:00, 80.9kB/s]


In [ ]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs

class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=use_cuda, activation='softplus1').to(device)
model.eval()

no input normalization


PreActResNet(
  (normalize): Normalize()
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer1): Sequential(
    (0): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (1): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
  )
  (layer2): Sequential(
    (0): PreActBloc

# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

In [ ]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True)
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        # TODO: Calculate the loss
        loss = ce_loss(output, labels)
        loss.backward()
        # TODO: compute the adv_x
        # find delta, clamp with eps
        adv_x = adv_x.detach() + eps_step * adv_x.grad.sign()
        delta = torch.clamp(adv_x - x, min=-eps, max=eps)
        adv_x = torch.clamp(x + delta, min=0, max=1).detach()

    return adv_x

In [ ]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True)
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        batch_size = x.size()[0]
        loss = ce_loss(output, labels)
        loss.backward()
        grad = adv_x.grad.data
        grad_norm = torch.norm(grad.view(batch_size, -1), p=2, dim=1)
        grad_norm = grad_norm.view(batch_size, 1, 1, 1)
        grad = grad / (grad_norm + 1e-10)
        adv_x = adv_x.detach() + eps_step * grad
        delta = adv_x - x
        delta_norm = torch.norm(delta.view(batch_size, -1), p=2, dim=1)
        delta_norm = delta_norm.view(batch_size, 1, 1, 1)
        delta = delta * eps / torch.clamp(delta_norm, min=eps)
        adv_x = torch.clamp(x + delta, min=0, max=1).detach()

    return adv_x

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [ ]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            x_adv = pgd_linf_untargeted(model, x_batch, y_batch, 10, eps, eps/4)
        elif attack == 'pgd_l2':
            x_adv = pgd_l2_untargeted(model, x_batch, y_batch, 10, eps, eps/4)
        else:
            pass

        tot_acc += (torch.max(model(x_adv), dim=1)[1] == y_batch).sum().item()
        tot_test += y_batch.size(0)

    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on {attack} attack with eps = {eps}')

## Multi-Norm Robust Accuracy

In [ ]:
def test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        x_adv_linf = pgd_linf_untargeted(model, x_batch, y_batch, 10, eps_linf, eps_linf/4)
        x_adv_l2 = pgd_l2_untargeted(model, x_batch, y_batch, 10, eps_l2, eps_l2/4)

        out = model(x_adv_linf)
        pred_linf = torch.max(out, dim=1)[1]
        out = model(x_adv_l2)
        pred_l2 = torch.max(out, dim=1)[1]

        tot_acc += ((pred_linf == y_batch) & (pred_l2 == y_batch)).sum().item()
        tot_test += y_batch.size(0)

    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on multi attacks')

# HW3 Problem 1


In [ ]:
TRAIN_EPSILONS = [8.0 / 255.0]
EVAL_EPSILONS = [2.0 / 255.0, 4.0 / 255.0, 8.0 / 255.0]

NUM_EPOCHS = 5
LEARNING_RATE = 0.1
WEIGHT_DECAY = 5e-4
TRAIN_PGD_STEPS = 5
EVAL_PGD_STEPS = 10

FORCE_RETRAIN = False
CHECKPOINT_DIR = "models/problem1_5epochs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def reset_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if use_cuda:
        torch.cuda.manual_seed_all(seed)

def build_model(seed=42):
    reset_seed(seed)
    return PreActResNet18(
        10, cuda=use_cuda, activation='softplus1'
    ).to(device)


In [ ]:
@torch.no_grad()
def evaluate_standard_accuracy(model, loader=test_loader):
    model.eval()
    correct, total = 0, 0

    for x_batch, y_batch in tqdm(loader, desc="Clean evaluation"):
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
        predictions = model(x_batch).argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    return correct / total


def evaluate_pgd_accuracy(model, eps, steps=EVAL_PGD_STEPS,
                          loader=test_loader):
    model.eval()
    correct, total = 0, 0
    step_size = eps / 4.0

    for x_batch, y_batch in tqdm(loader, desc=f"PGD evaluation eps={eps:.5f}"):
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        x_adv = pgd_linf_untargeted(
            model, x_batch, y_batch,
            k=steps, eps=eps, eps_step=step_size
        )

        with torch.no_grad():
            predictions = model(x_adv).argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    model.zero_grad(set_to_none=True)
    return correct / total


In [ ]:
def train_model(model, epochs, checkpoint_path, adversarial_eps=None,
                pgd_steps=TRAIN_PGD_STEPS):
    if os.path.exists(checkpoint_path) and not FORCE_RETRAIN:
        print(f"Loading existing checkpoint: {checkpoint_path}")
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        return model

    optimizer = optim.SGD(
        model.parameters(), lr=LEARNING_RATE,
        momentum=0.9, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{epochs}"
        )
        for x_batch, y_batch in progress:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            if adversarial_eps is None:
                training_x = x_batch
            else:
                training_x = pgd_linf_untargeted(
                    model, x_batch, y_batch,
                    k=pgd_steps,
                    eps=adversarial_eps,
                    eps_step=adversarial_eps / 4.0
                )
                model.train()

            optimizer.zero_grad(set_to_none=True)
            logits = model(training_x)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * y_batch.size(0)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total += y_batch.size(0)
            progress.set_postfix(
                loss=f"{running_loss / total:.4f}",
                accuracy=f"{100.0 * correct / total:.2f}%"
            )

        scheduler.step()
        print(
            f"Epoch {epoch + 1}: loss={running_loss / total:.4f}, "
            f"training accuracy={100.0 * correct / total:.2f}%"
        )

    torch.save(model.state_dict(), checkpoint_path)
    print(f"Saved checkpoint: {checkpoint_path}")
    return model


In [ ]:
baseline_path = os.path.join(CHECKPOINT_DIR, "standard_model.pth")
baseline_model = build_model(seed=42)
baseline_model = train_model(
    baseline_model,
    epochs=NUM_EPOCHS,
    checkpoint_path=baseline_path,
    adversarial_eps=None
)
del baseline_model
if use_cuda:
    torch.cuda.empty_cache()

adversarial_paths = {}
for eps in TRAIN_EPSILONS:
    eps_name = f"{int(round(eps * 255))}_255"
    checkpoint_path = os.path.join(
        CHECKPOINT_DIR, f"pgd_train_eps_{eps_name}.pth"
    )
    print(f"\nTraining adversarial model with epsilon={eps_name}")
    adversarial_model = build_model(seed=42)
    adversarial_model = train_model(
        adversarial_model,
        epochs=NUM_EPOCHS,
        checkpoint_path=checkpoint_path,
        adversarial_eps=eps,
        pgd_steps=TRAIN_PGD_STEPS
    )
    adversarial_paths[eps_name] = checkpoint_path
    del adversarial_model
    if use_cuda:
        torch.cuda.empty_cache()


no input normalization


Epoch 1/5: 100%|██████████| 782/782 [00:44<00:00, 17.66it/s, accuracy=36.37%, loss=1.7116]


Epoch 1: loss=1.7116, training accuracy=36.37%


Epoch 2/5: 100%|██████████| 782/782 [00:43<00:00, 17.80it/s, accuracy=52.15%, loss=1.3191]


Epoch 2: loss=1.3191, training accuracy=52.15%


...

Epoch 3: loss=1.8924, training accuracy=29.26%


Epoch 4/5: 100%|██████████| 782/782 [04:41<00:00,  2.78it/s, accuracy=31.06%, loss=1.8415]


Epoch 4: loss=1.8415, training accuracy=31.06%


Epoch 5/5: 100%|██████████| 782/782 [04:41<00:00,  2.78it/s, accuracy=32.26%, loss=1.7990]


Epoch 5: loss=1.7990, training accuracy=32.26%
Saved checkpoint: models/problem1_5epochs/pgd_train_eps_8_255.pth


## Part (a)


In [ ]:
model_paths = {"Standard training": baseline_path}
for eps_name, checkpoint_path in adversarial_paths.items():
    model_paths[f"PGD training eps={eps_name.replace('_', '/')}"] = checkpoint_path

part_a_rows = []
for model_name, checkpoint_path in model_paths.items():
    print(f"\nEvaluating {model_name}")
    evaluated_model = build_model(seed=42)
    evaluated_model.load_state_dict(
        torch.load(checkpoint_path, map_location=device)
    )

    row = {
        "Model": model_name,
        "Standard Accuracy (%)": 100.0 * evaluate_standard_accuracy(evaluated_model)
    }
    for eps in EVAL_EPSILONS:
        label = f"PGD Robust Accuracy eps={int(round(eps * 255))}/255 (%)"
        row[label] = 100.0 * evaluate_pgd_accuracy(
            evaluated_model, eps=eps, steps=EVAL_PGD_STEPS
        )

    part_a_rows.append(row)
    del evaluated_model
    if use_cuda:
        torch.cuda.empty_cache()

part_a_results = pd.DataFrame(part_a_rows)
part_a_results.to_csv("problem1a_results.csv", index=False)
part_a_results.round(2)



Evaluating Standard training
no input normalization


PGD evaluation eps=0.03137: 100%|██████████| 157/157 [01:37<00:00,  1.61it/s]



Evaluating PGD training eps=8/255
no input normalization


PGD evaluation eps=0.03137: 100%|██████████| 157/157 [01:36<00:00,  1.62it/s]


,Model,Standard Accuracy (%),PGD Robust Accuracy eps=2/255 (%),PGD Robust Accuracy eps=4/255 (%),PGD Robust Accuracy eps=8/255 (%)
0,Standard training,68.56,7.10,0.05,0.00
1,PGD training eps=8/255,49.60,45.15,40.77,31.74


## Part (b)


In [ ]:
def evaluate_fgsm_accuracy(model, eps, loader=test_loader):
    model.eval()
    correct, total = 0, 0
    for x_batch, y_batch in tqdm(loader, desc=f"FGSM evaluation eps={eps:.5f}"):
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
        x_adv = pgd_linf_untargeted(
            model, x_batch, y_batch, k=1, eps=eps, eps_step=eps
        )
        with torch.no_grad():
            predictions = model(x_adv).argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)
    model.zero_grad(set_to_none=True)
    return correct / total

part_b_rows = []
for model_name, checkpoint_path in model_paths.items():
    model = build_model(seed=42)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    for eps in EVAL_EPSILONS:
        part_b_rows.append({
            "Model": model_name,
            "Attack epsilon": f"{int(round(eps * 255))}/255",
            "PGD Accuracy (%)": 100.0 * evaluate_pgd_accuracy(
                model, eps=eps, steps=EVAL_PGD_STEPS
            ),
            "FGSM Accuracy (%)": 100.0 * evaluate_fgsm_accuracy(model, eps=eps)
        })
    del model
    if use_cuda:
        torch.cuda.empty_cache()

part_b_results = pd.DataFrame(part_b_rows)
part_b_results.to_csv("problem1b_results.csv", index=False)
part_b_results.round(2)


no input normalization


FGSM evaluation eps=0.03137: 100%|██████████| 157/157 [00:12<00:00, 12.74it/s]


no input normalization


FGSM evaluation eps=0.03137: 100%|██████████| 157/157 [00:12<00:00, 12.65it/s]


,Model,Attack epsilon,PGD Accuracy (%),FGSM Accuracy (%)
0,Standard training,2/255,7.10,19.16
1,Standard training,4/255,0.05,7.78
2,Standard training,8/255,0.00,7.32
3,PGD training eps=8/255,2/255,45.15,45.11
4,PGD training eps=8/255,4/255,40.77,40.74
5,PGD training eps=8/255,8/255,31.74,31.84
